# Module 04 — ML Theory & Evaluation
## 04-07: Bias-Variance Decomposition & ML Debugging

**Objective:** Formally derive the bias-variance-noise decomposition, implement Monte Carlo estimation from scratch, build learning curves as diagnostic tools, and apply CS229's systematic debugging recipe to diagnose and fix underfitting and overfitting.

**Prerequisites:** 4-06 (Learning Theory — VC Dimension, PAC Learning), 4-02 (Cross-Validation & Hyperparameter Tuning)

## Part 0 — Setup & Prerequisites

This notebook answers: *is my model underfitting or overfitting, and exactly what should I change?*

We will:
1. **Derive** the bias-variance-noise decomposition from the squared-error objective.
2. **Implement** Monte Carlo BV estimation from scratch — no library shortcuts.
3. **Plot** learning curves as the primary diagnostic (high bias vs high variance).
4. **Slice** errors by subgroup to identify systematic failure modes.
5. **Demonstrate** the double descent phenomenon that challenges classical BV intuition.

> **Theory notebook:** every theorem is validated with runnable experiments.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
import math
from typing import Any
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn

import random
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility & Device ─────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
N_POOL        = 1500   # regression training pool size for BV decomposition
N_TEST_REG    = 300    # regression test set size
TRAIN_SIZE    = 200    # bootstrap training size per Monte Carlo draw
N_BOOTSTRAP   = 250    # Monte Carlo bootstrap draws for BV estimation
NOISE_STD     = 0.30   # label noise std for the synthetic regression dataset
N_BINS        = 5      # quantile bins for error slicing
N_REPEATS_LC  = 8      # sub-sample repeats per training size in learning curves
N_CLS         = 2000   # classification dataset total samples
N_FEATURES    = 20     # total features in classification dataset
N_INFORMATIVE = 10     # informative features
N_TRAIN_DD    = 30     # training set size for double descent demonstration
DEGREE_MAX_DD = 55     # maximum polynomial degree for double descent sweep


In [ ]:
# ── Data Generation ──────────────────────────────────────────────────────────
# 1. Regression: f(x) = sin(2*pi*x) + 0.5*sin(4*pi*x) + N(0, NOISE_STD^2)
rng_data    = np.random.default_rng(SEED)
X_reg_all   = rng_data.uniform(0.0, 1.0, size=N_POOL)
y_reg_true  = (np.sin(2.0 * np.pi * X_reg_all)
               + 0.5 * np.sin(4.0 * np.pi * X_reg_all))
y_reg_noisy = y_reg_true + rng_data.normal(0.0, NOISE_STD, size=N_POOL)

test_idx_reg  = rng_data.choice(N_POOL, size=N_TEST_REG, replace=False)
pool_mask_reg = np.ones(N_POOL, dtype=bool)
pool_mask_reg[test_idx_reg] = False

X_te_reg      = X_reg_all[test_idx_reg].reshape(-1, 1)
y_te_reg_true = y_reg_true[test_idx_reg]
X_tr_pool_reg = X_reg_all[pool_mask_reg].reshape(-1, 1)
y_tr_pool_reg = y_reg_noisy[pool_mask_reg]

# 2. Binary classification (2000 samples, 20 features, 10 informative)
X_cls, y_cls = make_classification(
    n_samples=N_CLS, n_features=N_FEATURES, n_informative=N_INFORMATIVE,
    n_redundant=5, n_repeated=0, class_sep=0.8, flip_y=0.05,
    random_state=SEED,
)
X_tr_cls, X_te_cls, y_tr_cls, y_te_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=SEED, stratify=y_cls,
)

# 3. Moons dataset (non-linear, for debugging recipe demo)
X_moons, y_moons = make_moons(n_samples=600, noise=0.30, random_state=SEED)
X_tr_moon, X_te_moon, y_tr_moon, y_te_moon = train_test_split(
    X_moons, y_moons, test_size=0.3, random_state=SEED,
)

# 4. Small regression subset for double descent (n_train = N_TRAIN_DD)
idx_dd   = rng_data.choice(N_POOL, size=N_TRAIN_DD, replace=False)
X_tr_dd  = X_reg_all[idx_dd].reshape(-1, 1)
y_tr_dd  = y_reg_noisy[idx_dd]
X_te_dd  = X_te_reg
y_te_dd  = y_te_reg_true

print(f"Regression pool : {X_tr_pool_reg.shape[0]:>5d} train | {X_te_reg.shape[0]:>4d} test")
print(f"Classification  : {X_tr_cls.shape[0]:>5d} train | {X_te_cls.shape[0]:>4d} test"
      f"  (features={N_FEATURES}, informative={N_INFORMATIVE})")
print(f"Moons           : {X_tr_moon.shape[0]:>5d} train | {X_te_moon.shape[0]:>4d} test")
print(f"Double descent  : {X_tr_dd.shape[0]:>5d} train | {X_te_dd.shape[0]:>4d} test")


The datasets above span three learning regimes:
- **Regression pool** (1 200 pts): smooth function + Gaussian noise. We know $f(x)$
  without noise, so we can compute true Bias² and Variance.
- **Classification** (2 000 pts, 20 features): realistic binary task for learning curves.
- **Moons** (600 pts, non-linear): deliberately noisy boundary for the debugging recipe.
- **Small regression subset** (30 pts): intentionally tiny to trigger double descent.


---
## Part 1 — Theoretical Foundation & Implementation

### 1.1 The Bias-Variance-Noise Decomposition

For a regressor $\hat{f}$ trained on random dataset $\mathcal{D}$, the expected squared
loss at any test point $x$ decomposes as:

$$
\mathbb{E}_{\mathcal{D},\varepsilon}\!\left[(y - \hat{f}(x))^2\right]
= \underbrace{\bigl(\mathbb{E}_{\mathcal{D}}[\hat{f}(x)] - f(x)\bigr)^2}_{\text{Bias}^2}
+ \underbrace{\mathbb{E}_{\mathcal{D}}\!\left[\bigl(\hat{f}(x)
  - \mathbb{E}_{\mathcal{D}}[\hat{f}(x)]\bigr)^2\right]}_{\text{Variance}}
+ \underbrace{\sigma^2}_{\text{Noise}}
$$

**Proof sketch:** Let $\bar{f}(x) = \mathbb{E}_{\mathcal{D}}[\hat{f}(x)]$.
Add and subtract $\bar{f}(x)$ and $f(x)$ inside the squared error and expand.
Cross-terms vanish because $\mathbb{E}[\hat{f} - \bar{f}] = 0$ by definition,
and the noise $y - f(x)$ is independent of $\mathcal{D}$.

### 1.2 Component Interpretation

| Component | Formula | Cause | Fix |
|-----------|---------|-------|-----|
| **Bias²** | $(\bar{f}(x) - f(x))^2$ | Wrong inductive bias | More capacity |
| **Variance** | $\mathbb{E}[(\hat{f} - \bar{f})^2]$ | Sensitivity to $\mathcal{D}$ | More data / regularise |
| **Noise** $\sigma^2$ | $\mathbb{E}[(y - f)^2]$ | Inherent label noise | Cannot be reduced |

### 1.3 Monte Carlo Estimation

We estimate $\mathbb{E}_{\mathcal{D}}[\cdot]$ by:
1. Drawing $B$ bootstrap training sets $\mathcal{D}_1, \ldots, \mathcal{D}_B$.
2. Fitting a fresh estimator on each $\to$ predictions $\hat{f}_1(x), \ldots, \hat{f}_B(x)$.
3. Computing $\hat{\mu}(x) = \frac{1}{B}\sum_b \hat{f}_b(x)$, then:

$$
\widehat{\text{Bias}^2} = \frac{1}{n_{\text{test}}}\sum_i(\hat{\mu}(x_i) - f(x_i))^2,
\quad
\widehat{\text{Var}} = \frac{1}{n_{\text{test}}}\sum_i\operatorname{Var}_b[\hat{f}_b(x_i)]
$$


In [ ]:
# ── Part 1: Core BV Functions ─────────────────────────────────────────────────


def make_poly_pipeline(
    degree: int,
    alpha: float = 1e-3,
) -> Any:
    '''Build a polynomial Ridge regression sklearn Pipeline.

    Chains PolynomialFeatures -> StandardScaler -> Ridge ready for fit/predict.

    Args:
        degree: Polynomial feature expansion degree.
        alpha: Ridge regularisation strength.

    Returns:
        Unfitted sklearn Pipeline instance.
    '''
    return Pipeline([
        ("poly",   PolynomialFeatures(degree=degree, include_bias=False)),
        ("scaler", StandardScaler()),
        ("ridge",  Ridge(alpha=alpha)),
    ])


def bias_variance_decompose(
    estimator_factory: Any,
    X_pool: np.ndarray,
    y_pool: np.ndarray,
    X_test: np.ndarray,
    y_test_true: np.ndarray,
    n_bootstrap: int,
    train_size: int,
    seed: int,
) -> tuple[float, float]:
    '''Estimate bias^2 and variance via Monte Carlo bootstrapping.

    For each of n_bootstrap draws:
      - Sample train_size rows with replacement from (X_pool, y_pool).
      - Train a fresh estimator and predict on X_test.
    Aggregate statistics over bootstrap draws:
      bias^2   = mean_i  (mean_b[pred_b(x_i)] - y_true_i)^2
      variance = mean_i  Var_b[pred_b(x_i)]

    Args:
        estimator_factory: Callable returning a fresh sklearn-compatible estimator.
        X_pool: Training pool features, shape (n_pool, n_features).
        y_pool: Noisy training labels, shape (n_pool,).
        X_test: Test features, shape (n_test, n_features).
        y_test_true: Noiseless true labels at test points, shape (n_test,).
        n_bootstrap: Number of Monte Carlo bootstrap draws.
        train_size: Training examples per bootstrap draw.
        seed: Random seed for reproducibility.

    Returns:
        Tuple (bias_squared, variance) averaged over test points.
    '''
    rng    = np.random.default_rng(seed)
    n_test = len(X_test)
    preds  = np.zeros((n_bootstrap, n_test))
    for b in range(n_bootstrap):
        idx = rng.choice(len(X_pool), size=train_size, replace=True)
        est = estimator_factory()
        est.fit(X_pool[idx], y_pool[idx])
        preds[b] = est.predict(X_test)
    mean_pred = preds.mean(axis=0)
    bias_sq   = float(np.mean((mean_pred - y_test_true) ** 2))
    variance  = float(np.mean(preds.var(axis=0)))
    return bias_sq, variance


print("bias_variance_decompose defined.")
print(f"  B = {N_BOOTSTRAP} bootstrap draws | train_size = {TRAIN_SIZE}")


`make_poly_pipeline` and `bias_variance_decompose` are the fundamental building blocks.
`BiasVarianceDiagnostic` below wraps them into a reusable object with a structured
`.report()` interface, enabling clean comparison across multiple complexity levels.


In [ ]:
# ── BiasVarianceDiagnostic: Structured API ────────────────────────────────────


class BiasVarianceDiagnostic:
    '''Unified interface for bias-variance analysis on a regression dataset.

    Wraps bias_variance_decompose with persistent state and a structured
    report, enabling easy cross-model comparison at different complexities.

    Attributes:
        estimator_factory: Callable returning a fresh sklearn estimator.
        n_bootstrap: Number of Monte Carlo bootstrap draws.
        train_size: Bootstrap training set size.
        seed: Random seed.
        label: Human-readable name for this configuration.
        bias_sq_: Estimated bias^2 (set by fit()).
        variance_: Estimated variance (set by fit()).
    '''

    def __init__(
        self,
        estimator_factory: Any,
        n_bootstrap: int,
        train_size: int,
        seed: int,
        label: str = "model",
    ) -> None:
        '''Initialise BiasVarianceDiagnostic.

        Args:
            estimator_factory: Callable returning a fresh sklearn estimator.
            n_bootstrap: Number of Monte Carlo bootstrap draws.
            train_size: Bootstrap training set size.
            seed: Random seed for reproducibility.
            label: Human-readable label for this configuration.
        '''
        self.estimator_factory = estimator_factory
        self.n_bootstrap       = n_bootstrap
        self.train_size        = train_size
        self.seed              = seed
        self.label             = label
        self.bias_sq_: float | None  = None
        self.variance_: float | None = None

    def fit(
        self,
        X_pool: np.ndarray,
        y_pool: np.ndarray,
        X_test: np.ndarray,
        y_test_true: np.ndarray,
    ) -> "BiasVarianceDiagnostic":
        '''Run the Monte Carlo BV decomposition.

        Args:
            X_pool: Training pool features, shape (n_pool, n_features).
            y_pool: Noisy training labels, shape (n_pool,).
            X_test: Test features, shape (n_test, n_features).
            y_test_true: Noiseless true labels at test points, shape (n_test,).

        Returns:
            Self for method chaining.
        '''
        self.bias_sq_, self.variance_ = bias_variance_decompose(
            self.estimator_factory,
            X_pool, y_pool, X_test, y_test_true,
            self.n_bootstrap, self.train_size, self.seed,
        )
        return self

    def report(self) -> dict[str, Any]:
        '''Return a dictionary of computed BV metrics.

        Returns:
            Dict with keys: label, bias_squared, variance, total_reducible.

        Raises:
            RuntimeError: If fit() has not been called.
        '''
        if self.bias_sq_ is None or self.variance_ is None:
            raise RuntimeError("Call fit() before report().")
        return {
            "label":           self.label,
            "bias_squared":    round(self.bias_sq_, 5),
            "variance":        round(self.variance_, 5),
            "total_reducible": round(self.bias_sq_ + self.variance_, 5),
        }


print("BiasVarianceDiagnostic class defined.")


### 1.4 Sanity Check: Three Complexity Regimes

We verify the decomposition on degrees 1, 3, 5, 10, 15 — spanning high-bias through
high-variance. Expected: Bias² decreases and Variance increases monotonically with degree.


In [ ]:
# ── Sanity Check: Degree 1 (high bias), 5 (balanced), 15 (high variance) ──────
degrees_check = [1, 3, 5, 10, 15]

print("BV decomposition: true f(x)=sin(2*pi*x)+0.5*sin(4*pi*x) + noise")
print(f"  B={N_BOOTSTRAP} bootstrap draws | train_size={TRAIN_SIZE}")
print(f"{'Degree':>7s}  {'Bias^2':>9s}  {'Variance':>9s}  {'Total':>9s}  {'Dominant':>14s}")
print("-" * 58)

bvd_sanity_results: list[dict] = []
for deg_s in degrees_check:
    factory_s = lambda d=deg_s: make_poly_pipeline(d, alpha=1e-3)
    diag_s = BiasVarianceDiagnostic(
        factory_s, N_BOOTSTRAP, TRAIN_SIZE, SEED + deg_s, label=f"degree={deg_s}",
    )
    diag_s.fit(X_tr_pool_reg, y_tr_pool_reg, X_te_reg, y_te_reg_true)
    r_s = diag_s.report()
    bvd_sanity_results.append(r_s)
    dominant = "BIAS-dominated" if r_s["bias_squared"] > r_s["variance"] else "VAR-dominated"
    print(f"  {deg_s:5d}  {r_s['bias_squared']:9.5f}  {r_s['variance']:9.5f}"
          f"  {r_s['total_reducible']:9.5f}  {dominant:>14s}")

print("")
print("Degree  1: linear model cannot fit sin — systematically wrong (high bias).")
print("Degree  5: appropriate capacity, balanced bias and variance.")
print("Degree 15: memorises noise — predictions vary heavily per sample (high var).")


In [ ]:
# ── Visualise Regression Fits: 8 Bootstrap Samples per Degree ───────────────
fig_fits, axes_fits = plt.subplots(1, 3, figsize=(15, 4))
x_plot      = np.linspace(0.0, 1.0, 300).reshape(-1, 1)
y_plot_true = (np.sin(2.0 * np.pi * x_plot.ravel())
               + 0.5 * np.sin(4.0 * np.pi * x_plot.ravel()))

rng_viz      = np.random.default_rng(SEED + 42)
demo_degrees = [1, 5, 15]
demo_labels  = ['Degree 1 (high bias)', 'Degree 5 (balanced)', 'Degree 15 (high var)']

for ax_fit, deg_fit, label_fit in zip(axes_fits, demo_degrees, demo_labels):
    for _ in range(8):
        idx_fit  = rng_viz.choice(len(X_tr_pool_reg), size=TRAIN_SIZE, replace=True)
        pipe_fit = make_poly_pipeline(deg_fit, alpha=1e-3)
        pipe_fit.fit(X_tr_pool_reg[idx_fit], y_tr_pool_reg[idx_fit])
        ax_fit.plot(x_plot.ravel(), pipe_fit.predict(x_plot),
                    color='steelblue', alpha=0.28, lw=1.2)
    ax_fit.plot(x_plot.ravel(), y_plot_true, color='black', lw=2.5,
                label='True f(x)', ls='--')
    ax_fit.scatter(X_te_reg.ravel(), y_te_reg_true,
                   c='tomato', s=12, alpha=0.6, label='Test pts')
    ax_fit.set_title(label_fit, fontsize=10, fontweight='bold')
    ax_fit.set_xlabel('x')
    ax_fit.set_ylabel('y')
    ax_fit.legend(fontsize=7)
    ax_fit.set_ylim(-2.5, 2.5)

plt.suptitle('Bootstrap Fits: 8 draws per degree (blue) vs True Function (dashed black)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print('High-bias : all bootstrap fits cluster (low variance) but miss the target.')
print('High-var  : curves spread widely -- each fit memorises its sample differently.')


---
## Part 2 — Empirical Validation

Two experiments validate the theoretical BV predictions:

1. **BV vs model complexity** (degree 1 → 20): total error $= \text{Bias}^2 + \text{Var}$
   should form a U-shape with a minimum at the optimal degree.
2. **BV vs training set size** (fixed degree 10): adding data should reduce variance
   while leaving bias approximately constant — the core CS229 heuristic.

We then implement learning curves for classification — the practitioner's go-to diagnostic.


In [ ]:
# ── BV vs Polynomial Degree ───────────────────────────────────────────────────
degree_sweep = list(range(1, 21))
bias_arr     = np.zeros(len(degree_sweep))
var_arr      = np.zeros(len(degree_sweep))

print("Sweeping BV over polynomial degrees 1-20 (B=150 per degree) ...")
for i_d, deg_d in enumerate(degree_sweep):
    factory_d = lambda d=deg_d: make_poly_pipeline(d, alpha=1e-3)
    bsq_d, bvar_d = bias_variance_decompose(
        factory_d, X_tr_pool_reg, y_tr_pool_reg,
        X_te_reg, y_te_reg_true,
        n_bootstrap=150, train_size=TRAIN_SIZE, seed=SEED + i_d,
    )
    bias_arr[i_d] = bsq_d
    var_arr[i_d]  = bvar_d

noise_floor = float(NOISE_STD ** 2)
total_arr   = bias_arr + var_arr
optimal_deg = degree_sweep[int(np.argmin(total_arr))]

print(f"Optimal degree (min Bias^2+Var): {optimal_deg}")
print(f"Noise floor (sigma^2):           {noise_floor:.4f}")
print(f"Min total reducible error:       {total_arr.min():.4f}")
print("")

# BV vs training size at degree 10 (high-variance regime)
n_size_sweep = [50, 100, 150, 200, 300, 500, 750, 1000]
bias_n_arr   = np.zeros(len(n_size_sweep))
var_n_arr    = np.zeros(len(n_size_sweep))

print("Sweeping BV over training set sizes (degree=10, B=120 per size) ...")
factory_d10 = lambda: make_poly_pipeline(10, alpha=1e-3)
for j_n, n_tr_n in enumerate(n_size_sweep):
    n_tr_safe = min(n_tr_n, len(X_tr_pool_reg) - 1)
    bsq_n, bvar_n = bias_variance_decompose(
        factory_d10, X_tr_pool_reg, y_tr_pool_reg,
        X_te_reg, y_te_reg_true,
        n_bootstrap=120, train_size=n_tr_safe, seed=SEED + j_n + 100,
    )
    bias_n_arr[j_n] = bsq_n
    var_n_arr[j_n]  = bvar_n

print(f"{'n_train':>8s}  {'Bias^2':>9s}  {'Variance':>9s}  {'Total':>9s}")
print("-" * 40)
for j_n, n_tr_n in enumerate(n_size_sweep):
    print(f"  {n_tr_n:6d}  {bias_n_arr[j_n]:9.5f}  {var_n_arr[j_n]:9.5f}"
          f"  {bias_n_arr[j_n]+var_n_arr[j_n]:9.5f}")


In [ ]:
# ── BV vs n_train: Variance Reduction Rate Table ────────────────────────────
print('BV vs training size (degree=10): variance should decay as ~1/n')
print(f"{'n_train':>8s}  {'Bias^2':>9s}  {'Variance':>9s}  {'Total':>9s}  {'Var_ratio':>10s}")
print('-' * 52)
prev_var_n = None
for j_tab, n_tab in enumerate(n_size_sweep):
    var_val_tab  = float(var_n_arr[j_tab])
    bias_val_tab = float(bias_n_arr[j_tab])
    total_tab    = bias_val_tab + var_val_tab
    if prev_var_n is not None and prev_var_n > 0:
        ratio_str = f'{var_val_tab / prev_var_n:.3f}x'
    else:
        ratio_str = '---'
    print(f'  {n_tab:6d}  {bias_val_tab:9.5f}  {var_val_tab:9.5f}'
          f'  {total_tab:9.5f}  {ratio_str:>10s}')
    prev_var_n = var_val_tab
print('')
print('Theory: Var ~ 1/n. Doubling n reduces Var by roughly 0.5x.')
print('  Empirical ratios above should be close to 0.5 for well-behaved settings.')


The table above shows Variance shrinking as $n$ grows while Bias² stays approximately
flat — confirming the CS229 heuristic: *adding data reduces variance but not bias*.
The `Var_ratio` column quantifies this: theory predicts Var $\propto 1/n$,
so doubling $n$ should give a ratio near 0.5.


In [ ]:
# ── Visualise Bias-Variance Tradeoffs ──────────────────────────────────────────
fig_bv, axes_bv = plt.subplots(1, 2, figsize=(14, 5))

degrees_np  = np.array(degree_sweep)
axes_bv[0].plot(degrees_np, bias_arr,  label="Bias$^2$",    color="steelblue", lw=2)
axes_bv[0].plot(degrees_np, var_arr,   label="Variance",     color="tomato",    lw=2)
axes_bv[0].plot(degrees_np, total_arr, label="Bias$^2$+Var", color="purple",    lw=2.5, ls="--")
axes_bv[0].axhline(noise_floor, color="gray", ls=":", lw=1.5,
                   label=f"Noise ($\sigma^2={noise_floor:.3f}$)")
axes_bv[0].axvline(optimal_deg, color="green", ls="--", lw=1.8,
                   label=f"Optimal d={optimal_deg}")
axes_bv[0].set_xlabel("Polynomial Degree", fontsize=11)
axes_bv[0].set_ylabel("Error Component",   fontsize=11)
axes_bv[0].set_title("BV-Complexity Tradeoff (Monte Carlo, B=150 draws per degree)",
                     fontsize=11, fontweight="bold")
axes_bv[0].legend(fontsize=9)
axes_bv[0].set_ylim(bottom=0)

n_size_np = np.array(n_size_sweep)
axes_bv[1].plot(n_size_np, bias_n_arr,              label="Bias$^2$",    color="steelblue", lw=2, marker="o", ms=5)
axes_bv[1].plot(n_size_np, var_n_arr,               label="Variance",     color="tomato",    lw=2, marker="s", ms=5)
axes_bv[1].plot(n_size_np, bias_n_arr + var_n_arr,  label="Total",        color="purple",    lw=2.5, ls="--", marker="^", ms=5)
axes_bv[1].set_xlabel("Training Set Size (n)", fontsize=11)
axes_bv[1].set_ylabel("Error Component",       fontsize=11)
axes_bv[1].set_title("BV vs Training Size (degree=10) -- Variance falls; bias stays flat",
                     fontsize=11, fontweight="bold")
axes_bv[1].legend(fontsize=9)
axes_bv[1].set_ylim(bottom=0)

plt.suptitle("Bias-Variance Tradeoff: Empirical Validation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Confirmed: adding data reduces variance but NOT bias.")
print("  -> CS229 heuristic: large val-train gap + more data helps = variance problem.")
print("  -> High train error + more data does NOT help = bias problem.")


In [ ]:
# ── Regression Learning Curves: Degree 5 vs Degree 15 ───────────────────────
lc_reg_sizes = [50, 100, 200, 400, 600, 900, 1100]
rng_lc_reg   = np.random.default_rng(SEED + 77)
n_lc_test    = min(250, len(X_tr_pool_reg) - 1)
lc_test_idx  = rng_lc_reg.choice(len(X_tr_pool_reg), size=n_lc_test, replace=False)
lc_pool_mask = np.ones(len(X_tr_pool_reg), dtype=bool)
lc_pool_mask[lc_test_idx] = False
X_lc_pool = X_tr_pool_reg[lc_pool_mask]
y_lc_pool = y_tr_pool_reg[lc_pool_mask]
X_lc_test = X_tr_pool_reg[lc_test_idx]
# Approximate noiseless targets by using low-noise predictions
pipe_noiseless = make_poly_pipeline(20, alpha=1e-4)
pipe_noiseless.fit(X_tr_pool_reg, y_tr_pool_reg)
y_lc_test = pipe_noiseless.predict(X_lc_test)

lc5_tr  = np.zeros(len(lc_reg_sizes))
lc5_te  = np.zeros(len(lc_reg_sizes))
lc15_tr = np.zeros(len(lc_reg_sizes))
lc15_te = np.zeros(len(lc_reg_sizes))

for i_lr, n_lr in enumerate(lc_reg_sizes):
    n_safe_lr = min(n_lr, len(X_lc_pool) - 1)
    tr5_rep,  te5_rep  = [], []
    tr15_rep, te15_rep = [], []
    for _ in range(5):
        idx_lr   = rng_lc_reg.choice(len(X_lc_pool), size=n_safe_lr, replace=False)
        X_lr_tr  = X_lc_pool[idx_lr]
        y_lr_tr  = y_lc_pool[idx_lr]
        p5  = make_poly_pipeline(5,  alpha=1e-3)
        p5.fit(X_lr_tr, y_lr_tr)
        tr5_rep.append(float(np.mean((p5.predict(X_lr_tr)   - y_lr_tr)  ** 2)))
        te5_rep.append(float(np.mean((p5.predict(X_lc_test) - y_lc_test) ** 2)))
        p15 = make_poly_pipeline(15, alpha=1e-3)
        p15.fit(X_lr_tr, y_lr_tr)
        tr15_rep.append(float(np.mean((p15.predict(X_lr_tr)   - y_lr_tr)  ** 2)))
        te15_rep.append(float(np.mean((p15.predict(X_lc_test) - y_lc_test) ** 2)))
    lc5_tr[i_lr]  = float(np.mean(tr5_rep))
    lc5_te[i_lr]  = float(np.mean(te5_rep))
    lc15_tr[i_lr] = float(np.mean(tr15_rep))
    lc15_te[i_lr] = float(np.mean(te15_rep))

fig_lc_r, ax_lc_r = plt.subplots(figsize=(10, 4))
sz_lr = np.array(lc_reg_sizes)
ax_lc_r.semilogy(sz_lr, lc5_tr,  ls='--', color='steelblue', lw=2, label='Deg 5 (train)')
ax_lc_r.semilogy(sz_lr, lc5_te,  ls='-',  color='steelblue', lw=2, label='Deg 5 (val)')
ax_lc_r.semilogy(sz_lr, lc15_tr, ls='--', color='tomato',    lw=2, label='Deg 15 (train)')
ax_lc_r.semilogy(sz_lr, lc15_te, ls='-',  color='tomato',    lw=2, label='Deg 15 (val)')
ax_lc_r.set_xlabel('Training set size', fontsize=11)
ax_lc_r.set_ylabel('MSE (log scale)',   fontsize=11)
ax_lc_r.set_title('Regression Learning Curves (degree 5 vs 15)',
                  fontsize=11, fontweight='bold')
ax_lc_r.legend(fontsize=9)
plt.tight_layout()
plt.show()
print('Degree 5 : train-val gap closes with more data -- variance-dominated gap.')
print('Degree 15: gap persists even at n=1100 -- regularisation needed, not just data.')


### 2.2 Learning Curves — The Practitioner's Primary Diagnostic

BV decomposition requires the noiseless ground truth $f(x)$, unavailable in practice.
**Learning curves** (train and val error vs $n$) are the equivalent practitioner tool.

Key signatures to recognise:
- **High bias**: both curves are high, small gap, converge quickly — capacity problem.
- **High variance**: train error low, val error high — large persistent gap — data/regularise.
- **Well-fitted**: small gap that narrows smoothly as $n$ grows.


In [ ]:
# ── Learning Curve & Diagnosis Functions ──────────────────────────────────────


def compute_learning_curve(
    estimator_factory: Any,
    X: np.ndarray,
    y: np.ndarray,
    train_sizes: list[int],
    n_repeats: int,
    test_frac: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    '''Compute train and test error as a function of training set size.

    For each n in train_sizes and n_repeats random sub-samples:
      - Reserve test_frac of the data as a fixed hold-out.
      - Sample n training points from the pool.
      - Train a fresh estimator; record train and test error.
    Average over repeats to reduce curve variance.

    Args:
        estimator_factory: Callable() returning a fresh sklearn estimator.
        X: Feature matrix, shape (n_samples, n_features).
        y: Labels, shape (n_samples,).
        train_sizes: List of training set sizes to evaluate.
        n_repeats: Number of random sub-samples per training size.
        test_frac: Fraction of data held out as the fixed test set.
        seed: Random seed for reproducibility.

    Returns:
        Tuple (train_errors, val_errors), each np.ndarray of shape (len(train_sizes),).
    '''
    rng       = np.random.default_rng(seed)
    n_total   = len(X)
    n_test    = max(int(n_total * test_frac), 50)
    test_idx  = rng.choice(n_total, size=n_test, replace=False)
    pool_mask = np.ones(n_total, dtype=bool)
    pool_mask[test_idx] = False
    X_pool, y_pool = X[pool_mask], y[pool_mask]
    X_test,  y_test = X[test_idx],  y[test_idx]

    train_errs = np.zeros(len(train_sizes))
    val_errs   = np.zeros(len(train_sizes))
    for i_lc, n_tr_lc in enumerate(train_sizes):
        tr_rep: list[float] = []
        te_rep: list[float] = []
        for _ in range(n_repeats):
            idx_lc = rng.choice(len(X_pool), size=min(n_tr_lc, len(X_pool)), replace=False)
            X_tr_lc, y_tr_lc = X_pool[idx_lc], y_pool[idx_lc]
            est_lc = estimator_factory()
            est_lc.fit(X_tr_lc, y_tr_lc)
            tr_rep.append(float(np.mean(est_lc.predict(X_tr_lc) != y_tr_lc)))
            te_rep.append(float(np.mean(est_lc.predict(X_test)  != y_test)))
        train_errs[i_lc] = float(np.mean(tr_rep))
        val_errs[i_lc]   = float(np.mean(te_rep))
    return train_errs, val_errs


def diagnose_bias_variance(
    train_error: float,
    val_error: float,
    high_bias_threshold: float = 0.10,
    high_variance_gap: float   = 0.07,
) -> dict[str, Any]:
    '''Apply the CS229 bias-variance diagnostic to train/validation errors.

    Heuristic rules:
      - High bias     : train_error > high_bias_threshold.
      - High variance : val_error - train_error > high_variance_gap.

    Args:
        train_error: Training set 0-1 error rate (0 to 1).
        val_error: Validation / test 0-1 error rate (0 to 1).
        high_bias_threshold: Train error above this indicates high bias.
        high_variance_gap: Val-train gap above this indicates high variance.

    Returns:
        Dict with keys: diagnosis, train_error, val_error, gap, recommendations.
    '''
    gap          = val_error - train_error
    is_high_bias = train_error > high_bias_threshold
    is_high_var  = gap > high_variance_gap
    if is_high_bias and is_high_var:
        diagnosis = "High bias AND high variance"
        recs: list[str] = [
            "Increase model capacity (more features, deeper tree, larger network)",
            "Add more training data to reduce variance component",
            "Review feature engineering — model may lack necessary inputs",
        ]
    elif is_high_bias:
        diagnosis = "High bias (underfitting)"
        recs = [
            "Increase capacity: higher degree / more layers / larger C",
            "Add feature interactions or polynomial expansion",
            "Reduce regularisation strength",
            "Add more informative features",
        ]
    elif is_high_var:
        diagnosis = "High variance (overfitting)"
        recs = [
            "Collect more training data",
            "Increase regularisation (L2, reduce max_depth, dropout)",
            "Reduce model complexity or number of features",
            "Apply cross-validation for model selection",
        ]
    else:
        diagnosis = "Well-fitted"
        recs = [
            "Monitor performance on new data distributions",
            "Consider ensembles (bagging, boosting) for marginal gains",
        ]
    return {
        "diagnosis":       diagnosis,
        "train_error":     round(train_error, 4),
        "val_error":       round(val_error, 4),
        "gap":             round(gap, 4),
        "recommendations": recs,
    }


print("compute_learning_curve and diagnose_bias_variance defined.")


In [ ]:
# ── Learning Curves: Four Classifiers on Classification Dataset ───────────────
train_sizes_cls = [40, 80, 150, 250, 400, 600, 900, 1400]

clf_configs: list[dict] = [
    {
        "label":   "LR (high bias, C=0.001)",
        "color":   "steelblue",
        "factory": lambda: LogisticRegression(C=0.001, max_iter=500),
    },
    {
        "label":   "LR (balanced, C=1.0)",
        "color":   "green",
        "factory": lambda: LogisticRegression(C=1.0, max_iter=500),
    },
    {
        "label":   "DT (depth=3, constrained)",
        "color":   "orange",
        "factory": lambda: DecisionTreeClassifier(max_depth=3, random_state=SEED),
    },
    {
        "label":   "DT (depth=None, high variance)",
        "color":   "tomato",
        "factory": lambda: DecisionTreeClassifier(max_depth=None, random_state=SEED),
    },
]

print("Computing learning curves (4 classifiers x 8 sizes x 8 repeats) ...")
for cfg_lc in clf_configs:
    tr_errs_lc, te_errs_lc = compute_learning_curve(
        cfg_lc["factory"], X_cls, y_cls,
        train_sizes=train_sizes_cls,
        n_repeats=N_REPEATS_LC,
        test_frac=0.20,
        seed=SEED,
    )
    cfg_lc["train_errors"] = tr_errs_lc
    cfg_lc["val_errors"]   = te_errs_lc
    diag_lc = diagnose_bias_variance(float(tr_errs_lc[-1]), float(te_errs_lc[-1]))
    cfg_lc["diagnosis"] = diag_lc["diagnosis"]
    cfg_lc["recs"]      = diag_lc["recommendations"]
    print(f"  {cfg_lc['label']:<38s} | train={tr_errs_lc[-1]:.4f} | val={te_errs_lc[-1]:.4f}"
          f" | {diag_lc['diagnosis']}")

print("")
print("Learning curves computed. Plotting ...")


Each curve above is averaged over 8 random sub-samples per training size to smooth
the high variance of error estimates at small $n$. The shaded fill between train and
val curves shows the generalisation gap — wide shading = overfitting.


In [ ]:
# ── Visualise Learning Curves ──────────────────────────────────────────────────
fig_lc, axes_lc = plt.subplots(2, 2, figsize=(14, 10))
axes_lc_flat    = axes_lc.ravel()

for ax_lc, cfg_lc in zip(axes_lc_flat, clf_configs):
    sz_np = np.array(train_sizes_cls)
    ax_lc.plot(sz_np, cfg_lc["train_errors"], ls="--", color=cfg_lc["color"],
               lw=2, label="Train error", marker="o", ms=4)
    ax_lc.plot(sz_np, cfg_lc["val_errors"], ls="-",  color=cfg_lc["color"],
               lw=2, label="Val error",   marker="s", ms=4)
    ax_lc.fill_between(sz_np, cfg_lc["train_errors"], cfg_lc["val_errors"],
                       alpha=0.15, color=cfg_lc["color"])
    ax_lc.axhline(0.05, color="gray", ls=":", lw=1.2, label="5% target")
    ax_lc.set_title(cfg_lc["label"], fontsize=10, fontweight="bold")
    ax_lc.set_xlabel("Training Set Size", fontsize=9)
    ax_lc.set_ylabel("Error Rate",        fontsize=9)
    ax_lc.legend(fontsize=8)
    ax_lc.set_ylim(-0.01, 0.55)
    ax_lc.text(0.97, 0.97, cfg_lc["diagnosis"], transform=ax_lc.transAxes,
               ha="right", va="top", fontsize=7.5,
               bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8))

plt.suptitle("Learning Curves: High Bias vs High Variance", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("Reading the curves:")
print("  High-bias  : both errors HIGH, small gap, converge early — capacity problem.")
print("  High-var   : train LOW, val HIGH, large persistent gap — data/regularise.")
print("  Well-fitted: both errors low, small gap, val improves steadily with more data.")


---
## Part 3 — Application to Real Data

We apply the BV framework to two tasks:

1. **Error slicing on make_classification:** partition test errors by feature quantile bins
   to reveal systematic subgroup failures invisible in aggregate accuracy.

2. **CS229 debugging recipe on make_moons:** four classifiers spanning the bias-variance
   spectrum — diagnose each and produce structured recommendations.


In [ ]:
# ── Error Slicing: Subgroup Failure Analysis ───────────────────────────────────


def slice_error_analysis(
    X: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    feature_idx: int,
    feature_name: str,
    n_bins: int = 5,
) -> pd.DataFrame:
    '''Compute per-bin error rates by slicing on one feature's quantile bins.

    Divides the feature into n_bins equal-frequency quantile bins and computes
    per-bin error rate, sample count, and mean feature value. Reveals subgroup
    failures that overall accuracy hides — e.g., the model fails specifically
    for high values of a particular feature.

    Args:
        X: Feature matrix, shape (n_samples, n_features).
        y_true: True labels, shape (n_samples,).
        y_pred: Predicted labels, shape (n_samples,).
        feature_idx: Column index of the feature to slice on.
        feature_name: Human-readable name for the feature.
        n_bins: Number of quantile bins.

    Returns:
        DataFrame with columns [feature_range, n_samples, error_rate, mean_feature].
    '''
    feat_vals = X[:, feature_idx]
    quantiles = np.percentile(feat_vals, np.linspace(0, 100, n_bins + 1))
    rows: list[dict] = []
    for j_bin in range(n_bins):
        lo_bin, hi_bin = quantiles[j_bin], quantiles[j_bin + 1]
        if j_bin == n_bins - 1:
            mask_bin = (feat_vals >= lo_bin) & (feat_vals <= hi_bin)
        else:
            mask_bin = (feat_vals >= lo_bin) & (feat_vals < hi_bin)
        n_in_bin = int(mask_bin.sum())
        if n_in_bin == 0:
            continue
        err_rate_bin  = float(np.mean(y_pred[mask_bin] != y_true[mask_bin]))
        mean_feat_bin = float(feat_vals[mask_bin].mean())
        rows.append({
            f"{feature_name}_range": f"[{lo_bin:.2f}, {hi_bin:.2f}]",
            "n_samples":   n_in_bin,
            "error_rate":  round(err_rate_bin, 4),
            "mean_feature": round(mean_feat_bin, 4),
        })
    return pd.DataFrame(rows)


def cs229_debug_recipe(
    model_label: str,
    train_error: float,
    val_error: float,
) -> None:
    '''Print the CS229 structured debugging recipe for a given model.

    Applies diagnose_bias_variance and prints a formatted diagnostic report
    with numbered, actionable recommendations.

    Args:
        model_label: Human-readable name for the model configuration.
        train_error: Training set error rate (0 to 1).
        val_error: Validation / test error rate (0 to 1).

    Returns:
        None. Writes formatted report to stdout.
    '''
    result = diagnose_bias_variance(train_error, val_error)
    sep = "-" * 55
    print(sep)
    print(f"  Model      : {model_label}")
    print(f"  Train err  : {train_error:.4f}  |  Val err: {val_error:.4f}"
          f"  |  Gap: {result['gap']:.4f}")
    print(f"  Diagnosis  : {result['diagnosis']}")
    print("  Actions:")
    for k_rec, rec in enumerate(result["recommendations"], start=1):
        print(f"    {k_rec}. {rec}")


print("slice_error_analysis and cs229_debug_recipe defined.")


### 3.1 Error Slicing in Practice

Overall accuracy hides subgroup failures. Slicing by feature quantile reveals which
input regions the model struggles with — a mandatory step before deciding on a fix.
The feature with the largest error spread across bins is the most informative for debugging.


In [ ]:
# ── Error Slicing: Classification Dataset ─────────────────────────────────────
scaler_cls  = StandardScaler()
X_tr_cls_sc = scaler_cls.fit_transform(X_tr_cls)
X_te_cls_sc = scaler_cls.transform(X_te_cls)

lr_slice = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
lr_slice.fit(X_tr_cls_sc, y_tr_cls)
y_te_pred_cls = lr_slice.predict(X_te_cls_sc)

overall_err_cls = float(np.mean(y_te_pred_cls != y_te_cls))
print(f"Overall test error (LR C=1.0): {overall_err_cls:.4f}")
print("")

slice_features = [(0, "Feature_0"), (1, "Feature_1"), (4, "Feature_4"), (9, "Feature_9")]
error_spreads: list[tuple[str, float]] = []

for feat_idx_sl, feat_name_sl in slice_features:
    df_slice = slice_error_analysis(
        X_te_cls_sc, y_te_cls, y_te_pred_cls,
        feature_idx=feat_idx_sl, feature_name=feat_name_sl, n_bins=N_BINS,
    )
    spread_sl = float(df_slice["error_rate"].max() - df_slice["error_rate"].min())
    error_spreads.append((feat_name_sl, spread_sl))
    print(f"Sliced by {feat_name_sl} (error spread = {spread_sl:.4f}):")
    print(df_slice.to_string(index=False))
    print("")

print("Error spread ranking (highest = most informative feature for debugging):")
error_spreads.sort(key=lambda pair: pair[1], reverse=True)
for rank_i, (fname, spread_val) in enumerate(error_spreads, start=1):
    print(f"  {rank_i}. {fname}: spread = {spread_val:.4f}")


In [ ]:
# ── Multi-Model Error Slice Comparison: LR vs DT on Classification ──────────
# Train four classifiers and compare their error profiles across Feature_0 bins.
# This reveals whether different model types fail in the same input regions.

compare_clfs: list[dict] = [
    {'label': 'LR (C=0.001)',     'clf': LogisticRegression(C=0.001, max_iter=500)},
    {'label': 'LR (C=1.0)',       'clf': LogisticRegression(C=1.0,   max_iter=500)},
    {'label': 'DT (depth=3)',     'clf': DecisionTreeClassifier(max_depth=3,    random_state=SEED)},
    {'label': 'DT (depth=None)',  'clf': DecisionTreeClassifier(max_depth=None, random_state=SEED)},
]

print('Multi-model error slice comparison (Feature_0 quantile bins)')
print(f"{'Bin range':<22s}", end='')
for cfg_cmp in compare_clfs:
    print(f"  {cfg_cmp['label']:>17s}", end='')
print('')
print('-' * 92)

# Fit all classifiers on the full training set
for cfg_cmp in compare_clfs:
    cfg_cmp['clf'].fit(X_tr_cls_sc, y_tr_cls)
    cfg_cmp['y_pred'] = cfg_cmp['clf'].predict(X_te_cls_sc)

# Build per-bin error for each classifier
feat_vals_cmp = X_te_cls_sc[:, 0]
quantiles_cmp = [float(q) for q in np.percentile(feat_vals_cmp, np.linspace(0, 100, N_BINS + 1))]
bin_rows: list[dict] = []
for j_cmp in range(N_BINS):
    lo_cmp, hi_cmp = quantiles_cmp[j_cmp], quantiles_cmp[j_cmp + 1]
    if j_cmp == N_BINS - 1:
        mask_cmp = (feat_vals_cmp >= lo_cmp) & (feat_vals_cmp <= hi_cmp)
    else:
        mask_cmp = (feat_vals_cmp >= lo_cmp) & (feat_vals_cmp < hi_cmp)
    bin_label_cmp = f'[{lo_cmp:.2f}, {hi_cmp:.2f}]'
    row_cmp: dict = {'bin': bin_label_cmp, 'n': int(mask_cmp.sum())}
    for cfg_cmp in compare_clfs:
        err_cmp = float(np.mean(cfg_cmp['y_pred'][mask_cmp] != y_te_cls[mask_cmp]))
        row_cmp[cfg_cmp['label']] = round(err_cmp, 4)
    bin_rows.append(row_cmp)
    print(f"  {bin_label_cmp:<20s}", end='')
    for cfg_cmp in compare_clfs:
        print(f"  {row_cmp[cfg_cmp['label']]:>17.4f}", end='')
    print('')

print('')
print('Key: If two models fail in the SAME bins -> structural (feature) issue.')
print('     If they fail in DIFFERENT bins -> capacity/regularisation tradeoff.')

# Compute cross-model error correlation
err_matrix = np.array([
    [row_cmp[cfg_cmp['label']] for row_cmp in bin_rows]
    for cfg_cmp in compare_clfs
])
corr_matrix = np.corrcoef(err_matrix)
print('')
print('Error profile correlations across Feature_0 bins:')
print(f"{'':>17s}", end='')
for cfg_cmp in compare_clfs:
    print(f"  {cfg_cmp['label'][:17]:>17s}", end='')
print('')
for i_corr, cfg_i in enumerate(compare_clfs):
    print(f"  {cfg_i['label'][:17]:>17s}", end='')
    for j_corr in range(len(compare_clfs)):
        print(f"  {corr_matrix[i_corr, j_corr]:>17.3f}", end='')
    print('')
print('')
print('High correlation = models fail in the same input regions (structural).')
print('Low correlation  = different capacity creates different failure modes.')


### 3.2 CS229 Debugging Recipe — Structured Recommendations

Error slicing shows *where* the model fails. The CS229 recipe tells us *why* and *what to do*.
The recipe reduces to two questions:
1. Is **training error** too high? → **Capacity problem** (more features / layers / lower $\lambda$).
2. Is the **val-train gap** too large? → **Variance problem** (more data / stronger regularisation).


In [ ]:
# ── CS229 Debugging Recipe on make_moons ──────────────────────────────────────
scaler_moon  = StandardScaler()
X_tr_moon_sc = scaler_moon.fit_transform(X_tr_moon)
X_te_moon_sc = scaler_moon.transform(X_te_moon)

moons_configs: list[dict] = [
    {"label": "LR (C=0.01)",     "clf": LogisticRegression(C=0.01,  max_iter=1000)},
    {"label": "LR (C=1.0)",      "clf": LogisticRegression(C=1.0,   max_iter=1000)},
    {"label": "DT (depth=2)",    "clf": DecisionTreeClassifier(max_depth=2,    random_state=SEED)},
    {"label": "DT (depth=None)", "clf": DecisionTreeClassifier(max_depth=None, random_state=SEED)},
]

print("=" * 55)
print("CS229 Debugging Recipe — make_moons (noise=0.30)")
print("=" * 55)

for cfg_m in moons_configs:
    clf_m = cfg_m["clf"]
    clf_m.fit(X_tr_moon_sc, y_tr_moon)
    tr_err_m = float(np.mean(clf_m.predict(X_tr_moon_sc) != y_tr_moon))
    te_err_m = float(np.mean(clf_m.predict(X_te_moon_sc) != y_te_moon))
    cfg_m["train_error"] = tr_err_m
    cfg_m["val_error"]   = te_err_m
    cs229_debug_recipe(cfg_m["label"], tr_err_m, te_err_m)
    print("")


In [ ]:
# ── Structured Diagnosis Table: All Moons Configurations ────────────────────
diag_rows: list[dict] = []
for cfg_m2 in moons_configs:
    result_m2 = diagnose_bias_variance(cfg_m2['train_error'], cfg_m2['val_error'])
    diag_rows.append({
        'Model':     cfg_m2['label'],
        'Train_err': round(cfg_m2['train_error'], 4),
        'Val_err':   round(cfg_m2['val_error'],   4),
        'Gap':       result_m2['gap'],
        'Diagnosis': result_m2['diagnosis'],
    })
diag_df = pd.DataFrame(diag_rows)
print('Moons diagnosis summary:')
print(diag_df.to_string(index=False))
print('')
print('LR (C=0.01)    : linear cannot fit moons -- structural underfitting.')
print('LR (C=1.0)     : better C, same boundary shape -- bias is structural.')
print('DT (depth=2)   : limited splits -- mild bias, controlled variance.')
print('DT (depth=None): memorises training -- val-train gap is the giveaway.')


In [ ]:
# ── Decision Boundary Visualisation ──────────────────────────────────────────
fig_moon, axes_moon = plt.subplots(2, 2, figsize=(14, 10))
axes_moon_flat      = axes_moon.ravel()

h_mesh    = 0.04
x_lo, x_hi = X_te_moon_sc[:, 0].min() - 0.5, X_te_moon_sc[:, 0].max() + 0.5
y_lo, y_hi = X_te_moon_sc[:, 1].min() - 0.5, X_te_moon_sc[:, 1].max() + 0.5
xx_m, yy_m = np.meshgrid(np.arange(x_lo, x_hi, h_mesh), np.arange(y_lo, y_hi, h_mesh))

for ax_m, cfg_m in zip(axes_moon_flat, moons_configs):
    Z_m = cfg_m["clf"].predict(np.c_[xx_m.ravel(), yy_m.ravel()]).reshape(xx_m.shape)
    ax_m.contourf(xx_m, yy_m, Z_m, alpha=0.30, cmap="RdBu")
    ax_m.scatter(X_te_moon_sc[:, 0], X_te_moon_sc[:, 1],
                 c=y_te_moon, cmap="RdBu", edgecolors="k", s=18, lw=0.4)
    diag_txt = diagnose_bias_variance(cfg_m["train_error"], cfg_m["val_error"])["diagnosis"]
    title_m  = (f"{cfg_m['label']} | "
                f"train={cfg_m['train_error']:.3f} val={cfg_m['val_error']:.3f} | "
                f"{diag_txt}")
    ax_m.set_title(title_m, fontsize=8, fontweight="bold")
    ax_m.set_xlabel("Feature 0")
    ax_m.set_ylabel("Feature 1")

plt.suptitle("Decision Boundaries on make_moons (bias-variance spectrum)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("From left-right, top-bottom: underfitting -> balanced -> constrained -> overfitting.")


The decision boundaries above illustrate the four BV regimes visually:
- `LR (C=0.01)`: linear boundary — cannot fit the curved moons shape (high bias).
- `LR (C=1.0)`: still linear but better regularised — same structural limitation.
- `DT (depth=2)`: axis-aligned splits, limited depth — controlled variance, mild bias.
- `DT (depth=None)`: wiggly boundary memorising training noise — classic overfitting.


---
## Part 4 — Double Descent & Theory vs Practice

### 4.1 Double Descent

The classical BV picture predicts a U-shaped test error curve as complexity grows.
Modern deep learning violates this: with **overparameterised** models the test error can
*decrease again* beyond the interpolation threshold (where train error $\approx 0$).

$$
\text{Test error: } \underbrace{\searrow}_{\text{classical}}
\underbrace{\nearrow}_{\text{interp.\ threshold}}
\underbrace{\searrow}_{\text{overparameterised regime}}
$$

(Belkin et al., 2019): In the overparameterised regime, minimum-norm interpolating
solutions act as implicit regularisers.

### 4.2 Theory vs Empirical Check

We compare the Monte Carlo BV estimate $(\text{Bias}^2 + \text{Var})$ to the actual
test MSE for each polynomial degree, checking where the theoretical prediction holds.


In [ ]:
# ── Double Descent: Implementation ───────────────────────────────────────────


def double_descent_experiment(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    degree_max: int,
    alpha: float = 1e-8,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''Run polynomial regression of increasing degree to reveal double descent.

    Uses Ridge with near-zero alpha to approximate minimum-norm interpolation.
    Near the interpolation threshold (n_params ~ n_train) test error peaks;
    beyond it the overparameterised model generalises better again.

    Args:
        X_train: Training features, shape (n_train, 1).
        y_train: Training labels, shape (n_train,).
        X_test: Test features, shape (n_test, 1).
        y_test: Test labels (noiseless or noisy), shape (n_test,).
        degree_max: Maximum polynomial degree to evaluate.
        alpha: Ridge regularisation (near zero for near-interpolation behaviour).

    Returns:
        Tuple (degrees, train_mses, test_mses) each of shape (degree_max,).
    '''
    degrees_dd    = np.arange(1, degree_max + 1)
    train_mses_fn = np.zeros(degree_max)
    test_mses_fn  = np.zeros(degree_max)
    for i_dd, deg_dd in enumerate(degrees_dd):
        pipe_dd = make_poly_pipeline(deg_dd, alpha=alpha)
        pipe_dd.fit(X_train, y_train)
        train_mses_fn[i_dd] = float(np.mean((pipe_dd.predict(X_train) - y_train) ** 2))
        test_mses_fn[i_dd]  = float(np.mean((pipe_dd.predict(X_test)  - y_test)  ** 2))
    return degrees_dd, train_mses_fn, test_mses_fn


print("double_descent_experiment defined.")
print(f"  n_train (DD demo) : {N_TRAIN_DD}")
print(f"  Degree range      : 1 to {DEGREE_MAX_DD}")
print(f"  Interpolation threshold near degree {N_TRAIN_DD}")


In [ ]:
# ── Run and Visualise Double Descent ─────────────────────────────────────────
degrees_dd, train_mses_dd, test_mses_dd = double_descent_experiment(
    X_tr_dd, y_tr_dd, X_te_dd, y_te_dd,
    degree_max=DEGREE_MAX_DD, alpha=1e-8,
)

print(f"{'Degree':>7s}  {'Train MSE':>10s}  {'Test MSE':>10s}  {'Regime':>30s}")
print("-" * 65)
report_degrees = [1, 5, 10, 15, 20, 25, 29, 30, 31, 35, 40, 50, 55]
for deg_rep in report_degrees:
    if deg_rep > DEGREE_MAX_DD:
        break
    idx_rep = deg_rep - 1
    if deg_rep < N_TRAIN_DD:
        regime_str = "classical"
    elif deg_rep == N_TRAIN_DD:
        regime_str = "*** interpolation threshold ***"
    else:
        regime_str = "overparameterised"
    print(f"  {deg_rep:5d}  {train_mses_dd[idx_rep]:10.5f}  {test_mses_dd[idx_rep]:10.5f}"
          f"  {regime_str:>30s}")

peak_dd_idx  = int(np.argmax(test_mses_dd[:N_TRAIN_DD + 5]))
print("")
print(f"Peak test MSE at degree {degrees_dd[peak_dd_idx]}: {test_mses_dd[peak_dd_idx]:.4f}")
print(f"Test MSE at max degree {degrees_dd[-1]}:  {test_mses_dd[-1]:.4f}")


The table above shows the three regimes numerically. The log-scale plot below makes the
test MSE peak at the interpolation threshold (degree = n_train) clearly visible, and
shows the second descent that follows in the overparameterised regime.


In [ ]:
# ── Double Descent Plot ───────────────────────────────────────────────────────
fig_dd, ax_dd = plt.subplots(figsize=(12, 5))

ax_dd.semilogy(degrees_dd, train_mses_dd, ls="--", color="steelblue", lw=2,
               label="Train MSE")
ax_dd.semilogy(degrees_dd, test_mses_dd,  ls="-",  color="tomato",    lw=2.5,
               label="Test MSE")
ax_dd.axvline(N_TRAIN_DD, color="black", ls=":", lw=2.2,
              label=f"Interp. threshold (degree={N_TRAIN_DD})")
ax_dd.set_xlabel("Polynomial Degree (proxy for n_parameters)", fontsize=11)
ax_dd.set_ylabel("MSE  (log scale)",                           fontsize=11)
ax_dd.set_title(
    f"Double Descent: n_train={N_TRAIN_DD} | degree 1 to {DEGREE_MAX_DD}"
    " | Classical U-shape (left) + second descent (right of threshold)",
    fontsize=11, fontweight="bold",
)
ax_dd.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("Double descent confirmed:")
print("  Left of threshold : classical regime — bias falls, variance rises.")
print("  AT threshold      : train MSE ~ 0, test MSE peaks.")
print("  Right of threshold: overparameterised — test MSE decreases again.")
print("  Explanation: minimum-norm solutions in overparameterised regime act as")
print("  implicit regularisers that recover smooth fits from noisy labels.")


In [ ]:
# ── Theory vs Practice: BV Estimate vs Actual Test MSE ────────────────────────
print("Comparing Monte Carlo BV estimate to empirical test MSE (degrees 1-20)")
print(f"{'Degree':>7s}  {'Bias^2':>9s}  {'Variance':>9s}  {'BV_total':>9s}"
      f"  {'Test_MSE':>9s}  {'Abs_err':>9s}")
print("-" * 63)

test_mse_ref = np.zeros(len(degree_sweep))
for i_ref, deg_ref in enumerate(degree_sweep):
    pipe_ref = make_poly_pipeline(deg_ref, alpha=1e-3)
    pipe_ref.fit(X_tr_pool_reg, y_tr_pool_reg)
    preds_ref      = pipe_ref.predict(X_te_reg)
    test_mse_ref[i_ref] = float(np.mean((preds_ref - y_te_reg_true) ** 2))

abs_errs = np.abs(total_arr - test_mse_ref)
for i_ref, deg_ref in enumerate(degree_sweep):
    marker = " <-- close" if abs_errs[i_ref] < 0.03 else ""
    print(f"  {deg_ref:5d}  {bias_arr[i_ref]:9.5f}  {var_arr[i_ref]:9.5f}"
          f"  {total_arr[i_ref]:9.5f}  {test_mse_ref[i_ref]:9.5f}"
          f"  {abs_errs[i_ref]:9.5f}{marker}")

corr_bv = float(np.corrcoef(total_arr, test_mse_ref)[0, 1])
mae_bv  = float(np.mean(abs_errs))
print("")
print(f"Pearson r (BV_total vs Test_MSE): {corr_bv:.4f}")
print(f"Mean absolute error of BV estimate: {mae_bv:.5f}")
print("")
print("Where theory matches practice: low-to-moderate complexity (degrees 1-10).")
print("Divergence at high degree: finite bootstrap samples and Ridge regularisation")
print("  cause BV estimates to undercount test MSE — theory assumes infinite B.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Bias-variance decomposition** splits MSE into Bias² + Variance + Noise. Each
   component has a distinct cause and fix: Bias from wrong assumptions (fix: more capacity),
   Variance from sensitivity to training data (fix: more data or regularisation),
   Noise irreducible.

2. **Learning curves are the primary diagnostic.** High bias: both train and val error
   are high with a small gap. High variance: train error is low, val error is high — a
   persistent large gap that narrows with more data.

3. **CS229 debugging recipe:** check train error first. If high → capacity problem.
   If val-train gap is large → data or regularisation problem. Fix one thing at a time.

4. **Error slicing reveals subgroup failures.** A model with 90% overall accuracy may
   fail on 40% of a specific feature bin — always slice before deciding what to fix.

5. **Double descent challenges the classical U-curve.** Overparameterised models with
   minimum-norm solutions can generalise better than the interpolation threshold suggests —
   the classical BV picture is incomplete for modern deep learning.

### What's Next

→ **04-08 (Convex Optimization Foundations)** formalises Lagrangian duality and KKT
conditions — the tools that underpin the SVM dual and constrained RL policy optimisation.

### Going Further

- Belkin et al. (2019). *Reconciling modern ML practice and the BV trade-off.* PNAS.
- Geman et al. (1992). *Neural networks and the bias/variance dilemma.* Neural Computation.
- CS229 Lecture Notes (Andrew Ng) — Advice for Applying Machine Learning.


In [ ]:
# ── Final Summary ────────────────────────────────────────────────────────────
print("=" * 70)
print("04-07 BIAS-VARIANCE DECOMPOSITION & ML DEBUGGING — SUMMARY")
print("=" * 70)
print("")
print("1. BV Decomposition (Monte Carlo, degrees 1-20)")
optimal_d = int(np.argmin(total_arr)) + 1
print(f"   Optimal degree      : {optimal_d}")
print(f"   Min (Bias^2+Var)    : {total_arr.min():.4f}")
print(f"   Noise floor (sigma^2): {noise_floor:.4f}")
print(f"   Bias^2 (deg=1)  = {bias_arr[0]:.4f}  -- high bias (underfitting)")
print(f"   Bias^2 (deg=15) = {bias_arr[14]:.4f}  Var = {var_arr[14]:.4f}"  "  -- high variance")
print("")
print("2. Learning Curves (make_classification, 4 classifiers)")
print(f"   {'Config':<38s}  {'Train':>7s}  {'Val':>7s}  Diagnosis")
print(f"   {'-'*75}")
for cfg_pr in clf_configs:
    print(f"   {cfg_pr['label']:<38s}  {cfg_pr['train_errors'][-1]:7.4f}"  f"  {cfg_pr['val_errors'][-1]:7.4f}  {cfg_pr['diagnosis']}")
print("")
print("3. Double Descent (n_train=30, degree 1-55)")
peak_dd  = int(np.argmax(test_mses_dd))
print(f"   Interp. threshold: degree {N_TRAIN_DD}")
print(f"   Peak test MSE at degree {degrees_dd[peak_dd]}: {test_mses_dd[peak_dd]:.4f}")
print(f"   Test MSE at max degree  {degrees_dd[-1]}:  {test_mses_dd[-1]:.4f}")
print("")
print("4. Theory vs Practice")
print(f"   BV_total vs Test_MSE Pearson r : {corr_bv:.4f}")
print(f"   Mean absolute error of estimate: {mae_bv:.5f}")
print("")
print("=" * 70)
print("Debugging mantra (CS229):")
print("  High train error  -> CAPACITY problem  (more features / layers).")
print("  Large val-train gap -> DATA/REG problem (more data / regularise).")
print("=" * 70)
assert len(clf_configs) == 4, "Expected 4 classifier configurations."
assert corr_bv >= 0.5,        "BV-total should correlate with test MSE."
assert mae_bv  < 0.50,        "Mean BV estimation error should be reasonable."
print("All sanity assertions passed.")
